# QLoRA v4 — Full Qwen3.5-9B Agent Training on Colab

This is NOT the old 64-step v3E diagnostic. It trains a fresh 9B model with 4-bit NF4 all-linear LoRA on all 1,600 verified synthetic tool-agent training tasks. It cleans every deliberately failed first tool action, yielding 3,200 supervised next-action examples. Training is approximately one full epoch: 400 optimizer steps × 8 gradient accumulation, with checkpoints every 50 steps.

Scope limitation: the 1,600 tasks are a deterministic synthetic policy corpus, not a real-world multi-hop rollout corpus. The previous HotpotQA teacher pilot found only 2 deep clean trajectories out of 24 trials. Lower training loss does NOT establish better GAIA. Always compare GAIA100 and completion behavior after training.

Run in Google Colab with GPU selected. 9B QLoRA 2048-token all-linear training may need an L4 24GB or A100. A T4 may OOM. Do not silently switch model size or drop training actions.

Artifacts and intermediate checkpoints are saved under a user-owned Google Drive folder, with a frozen training contract and SHA256. The Colab user must authorize Drive and launch GPU. No GPU training has occurred just because this notebook has been created.

In [ ]:
import os, subprocess
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU first"
props=torch.cuda.get_device_properties(0)
VRAM=props.total_memory/1024**3
print({"gpu":props.name,"vram_gib":round(VRAM,2),"cuda":torch.version.cuda})
if VRAM < 20:
    raise RuntimeError("Full 9B all-linear QLoRA at context 2048 requires larger GPU memory. Select L4/A100; do not silently switch to a smaller model.")


In [ ]:
# Checkout audited full-train source, pinned independently of this notebook.
import subprocess,sys
from pathlib import Path
REPO=Path("/content/small-agent-qlora")
SOURCE_SHA="290c6400747110bfc1e0fed8ab0e06ea21ce3c58"
if not REPO.exists():
    subprocess.run(["git","clone","https://github.com/kolihhan/small-agent-qlora.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"fetch","--depth","100","origin","research/verified-teacher-trajectory-v0-20261112"],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",SOURCE_SHA],check=True)
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()==SOURCE_SHA
subprocess.run([sys.executable,"-m","pip","install","-q","-e",f"{REPO}[train,files,eval,dev]","jinja2>=3.1"],check=True)
print("AUDITED_SOURCE_PIN",SOURCE_SHA)


In [ ]:
# Regenerate all 1,600 verified synthetic train-only policy tasks.
import json,hashlib
from pathlib import Path
from gaia_small_agent.training.oracle_policy_v3 import generate_oracle_policy_v3_dataset
from scripts.prepare_full_qlora_v4 import prepare_full_clean_train

WORK=Path("/content/qlora-v4-preflight")
WORK.mkdir(exist_ok=True,parents=True)
generated=generate_oracle_policy_v3_dataset(WORK/"policy-v3",seed="small-agent-policy-v3")
source_rows=[json.loads(x) for x in generated["train"].read_text().splitlines() if x.strip()]
clean, report=prepare_full_clean_train(source_rows)
assert report["tasks"]==1600
assert report["removed_intentionally_bad_actions"]==400
assert report["total_trainable_actions"]==3200
assert len({x["task_id"] for x in clean})==1600
DATA_PATH=WORK/"v4-full-clean-1600-train.jsonl"
DATA_PATH.write_text("".join(json.dumps(x,sort_keys=True,ensure_ascii=False)+"\n" for x in clean))
DATA_SHA256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("FULL_CLEAN_DATA",json.dumps(report,sort_keys=True,indent=2),"sha256:",DATA_SHA256)


In [ ]:
# Real Qwen tokenizer; no silent truncation of history or tool observations.
from transformers import AutoTokenizer
from gaia_small_agent.training.qlora import _render_turn_examples,_tokenize_completion_examples
MODEL="Qwen/Qwen3.5-9B"
MAX_LENGTH=2048
tokenizer=AutoTokenizer.from_pretrained(MODEL,trust_remote_code=True)
train_actions=_render_turn_examples(tokenizer,clean)
assert len(train_actions)==3200
tokens=_tokenize_completion_examples(tokenizer,train_actions,
    max_length=MAX_LENGTH,require_full_prompt=True)
assert len(tokens)==3200
assert all(any(label!=-100 for label in x["labels"]) for x in tokens)
assert all(len(x["input_ids"])<=MAX_LENGTH for x in tokens)
print("REAL_9B_TOKENIZER_PREFLIGHT",{"actions":len(tokens),
    "max_sequence_tokens":max(len(x["input_ids"]) for x in tokens),
    "max_length":MAX_LENGTH,"prompts_full":True})


In [ ]:
# Save checkpoints and adapter persistently. Resume when Colab disconnects.
from google.colab import drive
drive.mount("/content/drive")
import json
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/small-agent-qlora/full-v4-3200")
ROOT.mkdir(exist_ok=True,parents=True)
contract={"source_sha":SOURCE_SHA,"model":MODEL,"data_sha256":DATA_SHA256,
          "tasks":1600,"supervised_actions":3200,"removed_bad_actions":400,
          "max_length":2048,"steps":400,"save_steps":50,"gradient_accumulation":8,
          "lora_target_modules":"all-linear","r":16,"alpha":32,
          "dropout":0.05,"lr":1e-4,"source":"synthetic oracle train split only"}
contract_file=ROOT/"train-contract.json"
if contract_file.exists():
    assert json.loads(contract_file.read_text())==contract, "Previously started run differs. Change ROOT instead."
else:
    contract_file.write_text(json.dumps(contract,indent=2,sort_keys=True))
(ROOT/"data-audit.json").write_text(json.dumps(report,indent=2,sort_keys=True))
print("Checkpoints/adapter folder:",ROOT)


In [ ]:
# All 3,200 clean actions, all-linear 9B QLoRA, 400 optimizer steps.
from transformers.trainer_utils import get_last_checkpoint
from gaia_small_agent.training.qlora import train_qlora
if (ROOT/"adapter_model.safetensors").exists():
    raise RuntimeError("A final adapter already exists; do not overwrite. Proceed to verification/evaluation.")
checkpoint=get_last_checkpoint(str(ROOT)) if ROOT.exists() else None
print("Resume checkpoint:",checkpoint)
train_qlora(
    data_path=DATA_PATH,
    output_dir=ROOT,
    model_name=MODEL,
    max_length=2048,
    epochs=1.0,
    learning_rate=1e-4,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    max_steps=400,
    save_steps=50,
    gradient_accumulation_steps=8,
    lora_target_modules="all-linear",
    exposure_count=None,
    require_full_prompt=True,
    resume_from_checkpoint=checkpoint,
)
print("TRAINING_COMPLETED: model output pending hash + step verification")


In [ ]:
# Reject partial or mislabeled runs; never claim success from loss/log messages alone.
import json,hashlib,zipfile
from pathlib import Path
from safetensors import safe_open
required=["adapter_model.safetensors","adapter_config.json","run_config.json"]
for name in required:
    assert (ROOT/name).is_file(), f"Missing output: {name}"
config=json.loads((ROOT/"run_config.json").read_text())
assert config["training_examples"]==3200
assert config["source_training_examples"]==3200
assert config["max_steps"]==400
assert config["max_length"]==2048
assert config["require_full_prompt"] is True
assert config["lora_target_modules"]=="all-linear"
assert config["gradient_accumulation_steps"]==8
adapter=ROOT/"adapter_model.safetensors"
assert adapter.stat().st_size>1_000_000
with safe_open(str(adapter),framework="pt",device="cpu") as weights:
    names=list(weights.keys())
    assert names and all("lora_" in name for name in names)
ADAPTER_SHA256=hashlib.sha256(adapter.read_bytes()).hexdigest()
(ROOT/"adapter-sha256.txt").write_text(ADAPTER_SHA256+"  adapter_model.safetensors\n")
print("FULL_V4_ADAPTER_VERIFIED",{"bytes":adapter.stat().st_size,
    "sha256":ADAPTER_SHA256,"training_examples":config["training_examples"],
    "optimizer_steps":config["max_steps"]})
print("IMPORTANT: This is an adapter artifact, NOT a GAIA accuracy result.")


## After Colab training

Use the repo's existing frozen matched GAIA Evaluation100 and synthetic held-out 200-task runner to compare v3B versus this full-v4 adapter, on the same runtime, same tooling, same random seed. Record correctness, completion, tool errors and duplicate calls. A lower supervised loss or improved synthetic performance alone does not justify promotion.

The adapter is a PEFT safetensors artifact. Convert to GGUF with the existing pinned llama.cpp workflow before reusing the frozen Ollama/llama.cpp GAIA runner. Do not evaluate by changing several runtime settings at once.